# 04 May 05 Default LayerNorm Scheduler/Epsilon Search

Keeps tuned LinearBidder bin clip fixed and searches only DQN scheduler + epsilon constants.

In [1]:
import sys
import json
import pickle
from dataclasses import replace
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = cwd
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.drlb.profiles import get_profile as get_drlb_profile
from example_notebooks.experiments.shared_runner import run_experiment_inprocess


/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
_linear_scr_fpa = Path(REPO_ROOT) / 'example_notebooks' / 'evaluate_baselines' / 'best_params' / 'fpa_baseline_n10_rndm_42' / 'linear_scr_FPA.pkl'
with _linear_scr_fpa.open('rb') as f:
    linear_tuned_params = pickle.load(f)
linear_tuned_params


{'coef': 0.023335213830958296,
 'lower_clip': 9,
 'upper_clip': 1,
 'factor': 3.4224852046754637}

In [3]:
RUN_NAME = 'may06_default_linear_clip_dqn_layer_norm_scheduler_epsilon_search'
DRLB_PROFILE = 'may05_default_linear_clip_dqn_layer_norm_scheduler_epsilon_search'
VERBOSE = False


In [ ]:
config = build_drlb_config(RUN_NAME, profile=DRLB_PROFILE, split_set='full_train_val_holdout')
config = replace(config, n_trials=10, refit_on='train_plus_val', max_steps=None)

profile_data = get_drlb_profile(DRLB_PROFILE)
base_drlb_params = dict(profile_data['base_drlb_params'])
reference_model_params = dict(profile_data['reference_model_params'])
base_drlb_params['bid_lower_clip'] = 3
base_drlb_params['bid_upper_clip'] = 8
base_drlb_params['traffic_path'] = str(REPO_ROOT / 'data' / 'traffic_share.csv')

result = run_experiment_inprocess(
    config,
    verbose=VERBOSE,
    base_drlb_params=base_drlb_params,
    reference_model_params=reference_model_params,
    state_type=profile_data['state_type'],
    objective=profile_data['objective'],
    search_space_fn=profile_data['search_space_fn'],
    n_trials=config.n_trials,
    max_train_steps=config.max_steps,
)

summary = result['summary']
{
    'run_name': config.run_name,
    'profile': DRLB_PROFILE,
    'linear_tuned_params': linear_tuned_params,
    'base_drlb_params': base_drlb_params,
    'reference_model_params': reference_model_params,
    'tuning_best_params': summary['tuning']['best_params'],
    'best_val_metrics': summary['tuning']['best_val_metrics'],
    'final_holdout_metrics': summary['final_holdout']['metrics'],
    'diagnostics_png': summary['refit']['combined_diagnostics_plot_path'],
}


In [ ]:
pd.DataFrame([
    {'artifact': 'run_summary_json', 'path': str(config.outputs_dir / 'run_summary.json')},
    {'artifact': 'metrics_json', 'path': str(config.outputs_dir / 'metrics.json')},
    {'artifact': 'drlb_diagnostics_png', 'path': str(config.outputs_dir / 'drlb_diagnostics.png')},
    {'artifact': 'best_refit_model', 'path': str(config.best_models_dir / 'best_refit.pt')},
])
